In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
CORR_DIR = Path("../reports/correlations")
CORR_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
profit_matrix = pd.read_csv("../data/processed/10m_default_2014_2025/profit_matrix.csv")
timestamps = pd.to_datetime(profit_matrix["date"])
profit_matrix["year"] = timestamps.dt.year
returns_cols = [col for col in profit_matrix.columns if col.endswith("_returns")]
corr = profit_matrix[returns_cols].corr(method="pearson")

In [ ]:
fig = plt.figure(figsize=(14, 10))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="rocket",
    square=True,
    linewidth=0.5,
    cbar_kws={"label": "Correlation"},
)
plt.title(
    "Correlation matrix of log_profitabilities 2014-2025",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout(pad=0.5)
plt.savefig(CORR_DIR / "avg_correlation_matrix.png", dpi=200)
plt.close(fig)

In [ ]:
years = sorted(profit_matrix["year"].unique())
corr_list = []

for year in years:
    subset = profit_matrix[profit_matrix["year"] == year]
    corr_matrix = subset[returns_cols].corr()
    avg_corr = corr_matrix.values[np.triu_indices_from(corr_matrix, k=1)].mean()
    corr_list.append({"year": year, "avg_corr": avg_corr})


corr_df = pd.DataFrame(corr_list)

fig = plt.figure(figsize=(15, 10))
plt.plot(
    corr_df["year"],
    corr_df["avg_corr"],
    marker="o",
    color="darkred",
    linewidth=1.2,
    markersize=8,
)
plt.axhline(
    y=corr_df["avg_corr"].mean(),
    color="red",
    linestyle="--",
    linewidth=1,
    label=f"Average correlation {corr_df['avg_corr'].mean():.2f}",
)
plt.title(
    "Average correlation between actions in period 2014-2025",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Average correlation", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
fig.savefig(CORR_DIR / "correlations_years.png", dpi=150)
plt.close(fig)

In [ ]:
years_list = [2014, 2019, 2022, 2025]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, year in enumerate(years_list):
    subset = profit_matrix[profit_matrix["year"] == year]
    corr_matrix = subset[returns_cols].corr()

    labels = [col.replace("_returns", "") for col in returns_cols]
    corr_matrix.index = labels
    corr_matrix.columns = labels

    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="rocket",
        square=True,
        cbar_kws={"label": "Correlation"},
        linewidth=0.5,
        ax=axes[idx],
    )
    axes[idx].set_title(
        f"Correlation matrix of {year} year", fontsize=12, fontweight="bold"
    )

plt.tight_layout()
fig.savefig(CORR_DIR / "corr_matrices_years.png", dpi=150)
plt.close(fig)